In [4]:
import re
from typing import Dict, List, Set, Tuple
from rdflib import Graph
import requests

TTL_FILE = "data/KB_construction/expanded_graph.ttl"
OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "gemma3:4b"
MAX_PREDICATES = 80
MAX_CLASSES = 40
SAMPLE_TRIPLES = 20
FORBIDDEN_PREFIXES = ("wd:", "wdt:", "wikibase:", "schema:", "brick:")
MY_NS = "http://myproject.org/ontology/"
LOCAL_ENTITY_CACHE: Dict[int, Set[str]] = {}

def ask_local_llm(prompt: str, model: str = GEMMA_MODEL) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(OLLAMA_URL, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Ollama API error {response.status_code} {response.text}")
    data = response.json()
    return data.get("response", "")

def load_graph(ttl_path: str) -> Graph:
    g = Graph()
    g.parse(ttl_path, format="turtle")
    print(f"Loaded {len(g)} triples from {ttl_path}")
    return g

def get_prefix_block(g: Graph) -> str:
    ns_map = {
        "my": "http://myproject.org/ontology/",
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "owl": "http://www.w3.org/2002/07/owl#"
    }
    for p, ns in g.namespace_manager.namespaces():
        ns_map[p] = str(ns)
        
    lines = [f"PREFIX {p}: <{ns}>" for p, ns in ns_map.items()]
    return "\n".join(sorted(lines))

def list_distinct_predicates(g: Graph, limit=MAX_PREDICATES) -> List[str]:
    q = f"SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [str(row.p) for row in g.query(q)]

def list_distinct_classes(g: Graph, limit=MAX_CLASSES) -> List[str]:
    q = f"SELECT DISTINCT ?cls WHERE {{ ?s a ?cls . }} LIMIT {limit}"
    return [str(row.cls) for row in g.query(q)]

def sample_triples(g: Graph, limit=SAMPLE_TRIPLES) -> List[Tuple[str, str, str]]:
    q = f"SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]

def sample_local_entities(g: Graph, limit=30) -> List[str]:
    q = f'''
    PREFIX my: <http://myproject.org/ontology/>
    SELECT DISTINCT ?s WHERE {{
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?s), STR(my:)))
    }}
    LIMIT {limit}
    '''
    entities = []
    for r in g.query(q):
        iri = str(r.s)
        if iri.startswith("http://myproject.org/ontology/"):
            entities.append(iri.replace("http://myproject.org/ontology/", "my:"))
    return entities

def build_schema_summary(g: Graph) -> str:
    prefixes = get_prefix_block(g)
    preds = list_distinct_predicates(g, limit=100)
    clss = list_distinct_classes(g, limit=50)
    samples = sample_triples(g, limit=40)
    entities = sample_local_entities(g, limit=120)
    
    pred_lines = "\n".join(f"- {p}" for p in preds)
    cls_lines = "\n".join(f"- {c}" for c in clss)
    sample_lines = "\n".join(f"- {s} {p} {o}" for s, p, o in samples)
    entity_lines = "\n".join(f"- {e}" for e in entities)
    
    hint = "Only local entities from namespace my: are valid. Never use wd:, wdt:, wikibase:, schema:, or brick:."
    
    summary = f"{prefixes}\n{hint}\n\nKnown local entities (sample):\n{entity_lines}\n\nPredicates:\n{pred_lines}\n\nClasses:\n{cls_lines}\n\nSample triples:\n{sample_lines}"
    return summary.strip()

SPARQL_INSTRUCTIONS = """
You are a SPARQL expert for a local Knowledge Graph.
Hard constraints (non-negotiable):
1. Use ONLY local entities with prefix my:.
2. Never use wd:, wdt:, wikibase:, schema:, brick:, or wikidata URLs.
3. If unsure about a predicate, prefer a safe pattern: my:Entity ?predicate ?object.
4. If the question mentions a person/organization, normalize spaces to underscores and use my:Name_Surname.
5. Return exactly one SPARQL 1.1 SELECT query and nothing else.

Examples:
Question: Who is Keith Gill?
SPARQL:
PREFIX my: <http://myproject.org/ontology/>
SELECT ?predicate ?object WHERE { my:Keith_Gill ?predicate ?object . }

Question: What is Melvin Capital?
SPARQL:
PREFIX my: <http://myproject.org/ontology/>
SELECT ?predicate ?object WHERE { my:Melvin_Capital ?predicate ?object . }

Question: List all organizations.
SPARQL:
PREFIX my: <http://myproject.org/ontology/>
SELECT ?subject WHERE { ?subject a my:ORG . }
# Do not require rdfs:label for organization listing queries.

Now convert the next question.
"""

def make_sparql_prompt(schema_summary: str, question: str) -> str:
    return f"{SPARQL_INSTRUCTIONS}\nSCHEMA SUMMARY\n{schema_summary}\nQUESTION\n{question}\nReturn only the SPARQL query in a code block"

CODE_BLOCK_RE = re.compile(r"`{3}(?:sparql)?\s*(.*?)`{3}", re.IGNORECASE | re.DOTALL)

def extract_sparql_from_text(text: str) -> str:
    m = CODE_BLOCK_RE.search(text)
    if m:
        return m.group(1).strip()
    return text.strip()

def ensure_prefixes(query: str) -> str:
    if re.search(r"^\s*PREFIX\s+", query, flags=re.IGNORECASE):
        return query
    return "PREFIX my: <http://myproject.org/ontology/>\n" + query

def contains_forbidden_namespace(query: str) -> bool:
    q = query.lower()
    if re.search(r"https?://(?:www\.)?wikidata\.org/", q):
        return True
    if re.search(r"(^|[\s<{(])(?:wd|wdt|wikibase|schema|brick):", q):
        return True
    return any(p in q for p in FORBIDDEN_PREFIXES)

def strict_regenerate_sparql(schema_summary: str, question: str, bad_query: str) -> str:
    prompt = (
        "The SPARQL below is invalid for this local graph because it uses forbidden namespaces. "
        "Rewrite it using only my: local entities and known predicates/classes from the schema summary. "
        "Never use wd:, wdt:, wikibase:, schema:, brick:, or wikidata URLs. "
        "Return one SPARQL 1.1 SELECT query in a code block.\n\n"
        f"SCHEMA SUMMARY\n{schema_summary}\n\nQUESTION\n{question}\n\nBAD QUERY\n{bad_query}"
    )
    raw = ask_local_llm(prompt)
    return extract_sparql_from_text(raw)

def normalize_entity_name(name: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_\-\s]", " ", name).strip()
    cleaned = re.sub(r"[\s\-]+", "_", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned

def extract_entity_from_question(question: str) -> str:
    patterns = [
        r"(?i)^\s*who\s+is\s+(.+?)\s*\??$",
        r"(?i)^\s*what\s+is\s+(.+?)\s*\??$",
        r"(?i)^\s*qui\s+est\s+(.+?)\s*\??$",
        r"(?i)^\s*qu[’']est\s*[- ]?ce\s+que\s+(.+?)\s*\??$"
    ]
    for p in patterns:
        m = re.match(p, question.strip())
        if m:
            return normalize_entity_name(m.group(1))
    quoted = re.search(r'"([^"]+)"', question)
    if quoted:
        return normalize_entity_name(quoted.group(1))
    return ""

def normalize_for_lookup(value: str) -> str:
    return re.sub(r"[^a-z0-9]", "", value.lower())

def get_local_entity_catalog(g: Graph, limit: int = 5000) -> Set[str]:
    cache_key = id(g)
    if cache_key in LOCAL_ENTITY_CACHE:
        return LOCAL_ENTITY_CACHE[cache_key]
    q = f'''
    PREFIX my: <http://myproject.org/ontology/>
    SELECT DISTINCT ?s WHERE {{
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?s), STR(my:)))
    }}
    LIMIT {limit}
    '''
    entities: Set[str] = set()
    for r in g.query(q):
        iri = str(r.s)
        if iri.startswith(MY_NS):
            entities.add(iri.replace(MY_NS, ""))
    LOCAL_ENTITY_CACHE[cache_key] = entities
    return entities

def extract_candidate_entity_phrases(question: str) -> List[str]:
    candidates: List[str] = []
    explicit = extract_entity_from_question(question)
    if explicit:
        candidates.append(explicit)
    quoted = re.findall(r'"([^"]+)"', question)
    for qv in quoted:
        candidates.append(normalize_entity_name(qv))
    cap_phrases = re.findall(r"\b([A-Z][A-Za-z0-9]+(?:[\s\-][A-Z][A-Za-z0-9]+)+)\b", question)
    for phrase in cap_phrases:
        candidates.append(normalize_entity_name(phrase))
    seen = set()
    ordered = []
    for c in candidates:
        if c and c not in seen:
            ordered.append(c)
            seen.add(c)
    return ordered

def choose_local_entity(g: Graph, question: str) -> str:
    catalog = get_local_entity_catalog(g)
    if not catalog:
        return ""
    normalized_catalog = {normalize_for_lookup(e): e for e in catalog}
    for candidate in extract_candidate_entity_phrases(question):
        key = normalize_for_lookup(candidate)
        if key in normalized_catalog:
            return normalized_catalog[key]
    for candidate in extract_candidate_entity_phrases(question):
        key = normalize_for_lookup(candidate)
        if not key:
            continue
        for norm, entity in normalized_catalog.items():
            if key in norm or norm in key:
                return entity
    return ""

def patch_forbidden_query_with_local_entity(g: Graph, question: str, query: str) -> str:
    entity = choose_local_entity(g, question)
    if not entity:
        return ""
    patched = re.sub(r"\b(?:wd|wdt):[A-Za-z0-9_]+", f"my:{entity}", query)
    patched = re.sub(r"<https?://(?:www\.)?wikidata\.org/(?:entity|prop/direct)/[^>]+>", f"my:{entity}", patched)
    patched = re.sub(r"^\s*PREFIX\s+(?:wd|wdt|wikibase|schema|brick):[^\n]*\n?", "", patched, flags=re.IGNORECASE | re.MULTILINE)
    return ensure_prefixes(patched)

def build_entity_fallback_query(g: Graph, question: str) -> str:
    entity = choose_local_entity(g, question) or extract_entity_from_question(question)
    if not entity:
        return ""
    return (
        "PREFIX my: <http://myproject.org/ontology/>\n"
        f"SELECT ?predicate ?object WHERE {{ my:{entity} ?predicate ?object . }}"
    )

def is_organization_listing_question(question: str) -> bool:
    q = question.lower()
    patterns = [
        r"\blist\s+all\s+organizations?\b",
        r"\blist\s+organizations?\b",
        r"\bshow\s+all\s+organizations?\b",
        r"\btoutes?\s+les\s+organisations\b",
        r"\blister\s+les\s+organisations\b",
    ]
    return any(re.search(p, q) for p in patterns)

def build_organization_fallback_query() -> str:
    return (
        "PREFIX my: <http://myproject.org/ontology/>\n"
        "SELECT DISTINCT ?organization WHERE {\n"
        "  { ?organization a my:ORG . }\n"
        "  UNION\n"
        "  { ?organization a my:Organization . }\n"
        "}"
    )

def try_organization_fallback(g: Graph, question: str) -> dict:
    if not is_organization_listing_question(question):
        return {}
    q = build_organization_fallback_query()
    try:
        vars_, rows = run_sparql(g, q)
        if rows:
            return {"query": q, "vars": vars_, "rows": rows, "repaired": True, "fallback": "organization_list", "error": None}
    except Exception:
        pass
    return {}

def generate_sparql(question: str, schema_summary: str) -> str:
    raw = ask_local_llm(make_sparql_prompt(schema_summary, question))
    query = extract_sparql_from_text(raw)
    return ensure_prefixes(query)

def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
    res = g.query(query)
    vars_ = [str(v) for v in res.vars]
    rows = [tuple(str(cell) for cell in r) for r in res]
    return vars_, rows

REPAIR_INSTRUCTIONS = "The previous SPARQL failed to execute. Using the schema summary and the error message, return a corrected SPARQL 1.1 SELECT query. Use only local namespace my: and known prefixes. Never use wd:, wdt:, wikibase:, schema:, brick:, or wikidata URLs. Return only one code block."

def repair_sparql(schema_summary: str, question: str, bad_query: str, error_msg: str) -> str:
    prompt = f"{REPAIR_INSTRUCTIONS}\nSCHEMA SUMMARY\n{schema_summary}\nORIGINAL QUESTION\n{question}\nBAD SPARQL\n{bad_query}\nERROR MESSAGE\n{error_msg}\nReturn only the corrected SPARQL in a code block"
    raw = ask_local_llm(prompt)
    return ensure_prefixes(extract_sparql_from_text(raw))

def answer_with_sparql_generation(g: Graph, schema_summary: str, question: str, try_repair: bool = True) -> dict:
    sparql = generate_sparql(question, schema_summary)
    if contains_forbidden_namespace(sparql):
        sparql = ensure_prefixes(strict_regenerate_sparql(schema_summary, question, sparql))
    if contains_forbidden_namespace(sparql):
        patched = patch_forbidden_query_with_local_entity(g, question, sparql)
        if patched and not contains_forbidden_namespace(patched):
            sparql = patched
        else:
            org_fallback = try_organization_fallback(g, question)
            if org_fallback:
                return org_fallback
            fallback_query = build_entity_fallback_query(g, question)
            if fallback_query:
                try:
                    vars_, rows = run_sparql(g, fallback_query)
                    return {"query": fallback_query, "vars": vars_, "rows": rows, "repaired": True, "fallback": "entity", "error": None}
                except Exception as e:
                    return {"query": fallback_query, "vars": [], "rows": [], "repaired": True, "fallback": "entity", "error": str(e)}
    try:
        vars_, rows = run_sparql(g, sparql)
        if rows:
            return {"query": sparql, "vars": vars_, "rows": rows, "repaired": False, "fallback": None, "error": None}
        org_fallback = try_organization_fallback(g, question)
        if org_fallback:
            return org_fallback
        fallback_query = build_entity_fallback_query(g, question)
        if fallback_query and fallback_query != sparql:
            try:
                f_vars, f_rows = run_sparql(g, fallback_query)
                if f_rows:
                    return {"query": fallback_query, "vars": f_vars, "rows": f_rows, "repaired": True, "fallback": "entity", "error": None}
            except Exception:
                pass
        return {"query": sparql, "vars": vars_, "rows": rows, "repaired": False, "fallback": None, "error": None}
    except Exception as e:
        err = str(e)
        if try_repair:
            repaired = repair_sparql(schema_summary, question, sparql, err)
            if contains_forbidden_namespace(repaired):
                patched = patch_forbidden_query_with_local_entity(g, question, repaired)
                if patched and not contains_forbidden_namespace(patched):
                    repaired = patched
                else:
                    org_fallback = try_organization_fallback(g, question)
                    if org_fallback:
                        return org_fallback
                    fallback_query = build_entity_fallback_query(g, question)
                    if fallback_query:
                        try:
                            vars_, rows = run_sparql(g, fallback_query)
                            return {"query": fallback_query, "vars": vars_, "rows": rows, "repaired": True, "fallback": "entity", "error": None}
                        except Exception as e2:
                            return {"query": fallback_query, "vars": [], "rows": [], "repaired": True, "fallback": "entity", "error": str(e2)}
            try:
                vars_, rows = run_sparql(g, repaired)
                if rows:
                    return {"query": repaired, "vars": vars_, "rows": rows, "repaired": True, "fallback": None, "error": None}
                org_fallback = try_organization_fallback(g, question)
                if org_fallback:
                    return org_fallback
                fallback_query = build_entity_fallback_query(g, question)
                if fallback_query and fallback_query != repaired:
                    try:
                        f_vars, f_rows = run_sparql(g, fallback_query)
                        if f_rows:
                            return {"query": fallback_query, "vars": f_vars, "rows": f_rows, "repaired": True, "fallback": "entity", "error": None}
                    except Exception:
                        pass
                return {"query": repaired, "vars": vars_, "rows": rows, "repaired": True, "fallback": None, "error": None}
            except Exception as e2:
                org_fallback = try_organization_fallback(g, question)
                if org_fallback:
                    return org_fallback
                fallback_query = build_entity_fallback_query(g, question)
                if fallback_query:
                    try:
                        vars_, rows = run_sparql(g, fallback_query)
                        return {"query": fallback_query, "vars": vars_, "rows": rows, "repaired": True, "fallback": "entity", "error": None}
                    except Exception:
                        pass
                return {"query": repaired, "vars": [], "rows": [], "repaired": True, "fallback": None, "error": str(e2)}
        else:
            return {"query": sparql, "vars": [], "rows": [], "repaired": False, "fallback": None, "error": err}

def answer_no_rag(question: str) -> str:
    prompt = f"Answer the following question as best as you can\n\n{question}"
    return ask_local_llm(prompt)

def pretty_print_result(result: dict):
    if result.get("error"):
        print("Execution Error", result["error"])
    print("SPARQL Query Used")
    print(result["query"])
    print("Repaired", result["repaired"])
    if result.get("fallback"):
        print("Fallback", result["fallback"])
    vars_ = result.get("vars", [])
    rows = result.get("rows", [])
    if not rows:
        print("No rows returned")
        return
    print("Results")
    print(" | ".join(vars_))
    for r in rows[:20]:
        print(" | ".join(r))
    if len(rows) > 20:
        print(f"showing 20 of {len(rows)}")

if __name__ == "__main__":
    g = load_graph(TTL_FILE)
    schema = build_schema_summary(g)
    while True:
        q = input("Question or type quit to exit: ").strip()
        if q.lower() == "quit":
            break
        print("Baseline No RAG")
        print(answer_no_rag(q))
        print("\n\nSPARQL generation RAG")
        result = answer_with_sparql_generation(g, schema, q, try_repair=True)
        pretty_print_result(result)

Loaded 128344 triples from data/KB_construction/expanded_graph.ttl
Baseline No RAG
Okay, you've asked a *massive* question! "List all organizations" is impossible. There are literally billions of organizations on the planet, from tiny local groups to multinational corporations. 

Instead of a complete, unmanageable list, I'll give you a breakdown categorized by type and size, with examples. This will give you a sense of the breadth of organizations out there.

**I. Government & Political Organizations:**

* **National Governments:** (Examples: United States Government, United Kingdom Government, Canadian Government, etc. - Each has countless agencies and departments)
* **Local Governments:** (Cities, Counties, Municipalities - Again, many sub-organizations)
* **Political Parties:** (Democratic Party, Republican Party, Conservative Party, Labour Party, Green Party, etc. – each with numerous local branches)
* **International Organizations:**
    * **United Nations (UN):**  The largest an